# Feature-wise EDA

In [1]:
import os
from dotenv import load_dotenv
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import numpy as np

from statsmodels.graphics.tsaplots import plot_acf

from impulse.collection.database import ImpulseDB
from impulse.collection.s3_manager import S3Manager
from impulse.replay_dataset import ReplayDataset

load_dotenv()

True

In [2]:
impulse_path = os.getenv('IMPULSE_PATH')
db_path = os.getenv('DB_PATH')
project_root = os.getenv('PROJECT_ROOT')

db = ImpulseDB(db_path)
s3 = S3Manager()

Database initialized: /Users/david/dev/impulse/impulse.db
✓ S3 Manager initialized (Local Credentials)
  Region: us-east-2
  Bucket: impulse-rl-4e8e-a163-3a3b24a95b2b


## The dataset (sample of 200 replays)

In [3]:
cache_dir = project_root+'replay-cache/'
dataset = ReplayDataset(
    db_path = db_path,
    s3_manager = s3,
    cache_dir=cache_dir
)

replay_list = dataset.load_sample(200, seed=51)

Found 7282 parsed replays in database
Loaded 200/200 replays


In [4]:
type(replay_list)

list

In [5]:
type(replay_list[0])

impulse.replay_dataset.ReplayData

In [6]:
replay_list[0].__dict__.keys()

dict_keys(['replay_id', 'frames', 'metadata'])

In [7]:
replay_list[0].replay_id

'98a9996a-fd49-46e0-8be0-95f499042d6d'

In [8]:
replay_list[0].metadata

{'replay_id': '98a9996a-fd49-46e0-8be0-95f499042d6d',
 'frame_count': 9394,
 'feature_count': 125,
 'fps': 30.0,
 'parsed_at': '2026-03-10T21:08:48.008620+00:00',
 'source_file': '/tmp/tmpbuy8t0h3/98a9996a-fd49-46e0-8be0-95f499042d6d.replay',
 'ballchasing_id': '26826DBB47360884FFC22D89ED4A3A41',
 'replay_name': 'NA P-G G2 vs SSG G1 2024-05-12.14.30',
 'date': '2024-05-12 14-30-15',
 'map': 'EuroStadium_Night_P',
 'match_type': 'Private',
 'team_size': 3,
 'num_frames': 9269,
 'duration_seconds': 313.1333333333333,
 'team_0_score': 1,
 'team_1_score': 2,
 'goals': [{'PlayerName': 'hockser', 'PlayerTeam': 1, 'frame': 3534},
  {'PlayerName': 'Daniel', 'PlayerTeam': 0, 'frame': 5372},
  {'PlayerName': 'hockser', 'PlayerTeam': 1, 'frame': 6301}],
 'highlights': [{'BallName': 'Ball_TA_137',
   'CarName': 'Car_TA_882',
   'GoalActorName': 'None',
   'frame': 2415},
  {'BallName': 'Ball_TA_137',
   'CarName': 'Car_TA_884',
   'GoalActorName': 'None',
   'frame': 3217},
  {'BallName': 'Ball_TA

## Feature summary statistics 

In [9]:
frames_list = [replay.frames for replay in replay_list]
len(frames_list)

200

In [10]:
all_frames = pd.concat(frames_list, ignore_index=False)
all_frames.shape

(1992567, 125)

In [11]:
all_frames.describe()

,frame,current time,frame time,seconds remaining,Ball - position x,Ball - position y,Ball - position z,Ball - linear velocity x,Ball - linear velocity y,Ball - linear velocity z,...,p5_angular velocity z,p5_quaternion x,p5_quaternion y,p5_quaternion z,p5_quaternion w,p5_boost level,p5_dodge active,p5_jump active,p5_double jump active,p5_player demolished by
count,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,...,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06,1.992567e+06
mean,5.140915e+03,2.441255e+02,2.441439e+02,1.452991e+02,-8.440375e+01,6.140936e+01,7.819397e+02,-8.668629e-01,5.805315e+00,3.978612e+00,...,2.126430e+00,6.236118e-02,3.701597e-02,3.486357e-01,3.514476e-01,1.153211e+02,1.877439e+01,2.330430e+01,4.986069e+00,-9.957201e-01
std,3.283787e+03,1.734064e+02,1.734065e+02,9.005945e+01,2.427756e+03,3.001414e+03,5.281395e+02,1.139089e+03,1.362263e+03,6.094742e+02,...,1.640690e+02,2.749680e-01,2.811851e-01,5.468687e-01,5.440048e-01,8.964184e+01,1.541249e+01,1.946405e+01,4.921458e+00,9.705030e-02
min,0.000000e+00,1.133783e+01,1.137112e+01,0.000000e+00,-4.027560e+03,-5.215400e+03,7.308000e+01,-4.731410e+03,-4.429510e+03,-3.129310e+03,...,-5.499900e+02,-7.070960e-01,-7.070960e-01,-7.071068e-01,-7.071068e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00
25%,2.490000e+03,1.285239e+02,1.285422e+02,6.700000e+01,-2.275670e+03,-2.598515e+03,3.096400e+02,-7.625500e+02,-1.049840e+03,-3.765600e+02,...,-9.778000e+01,-4.674591e-03,-2.851150e-03,-1.361841e-01,-1.297804e-01,3.100000e+01,8.000000e+00,1.000000e+01,0.000000e+00,-1.000000e+00
50%,4.981000e+03,2.294389e+02,2.294552e+02,1.430000e+02,-1.073000e+01,0.000000e+00,7.033600e+02,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-2.225354e-03,3.288128e-03,5.243196e-01,5.274499e-01,1.080000e+02,1.400000e+01,1.800000e+01,4.000000e+00,-1.000000e+00
75%,7.476000e+03,3.317843e+02,3.318006e+02,2.210000e+02,2.080880e+03,2.743690e+03,1.190430e+03,7.591850e+02,1.059490e+03,3.754700e+02,...,1.015100e+02,2.529898e-02,4.814856e-03,8.365358e-01,8.349700e-01,2.000000e+02,2.600000e+01,3.200000e+01,8.000000e+00,-1.000000e+00
max,2.598200e+04,2.005320e+03,2.005346e+03,5.630000e+02,4.028630e+03,5.215430e+03,1.973510e+03,4.636870e+03,4.187040e+03,4.811520e+03,...,5.499900e+02,9.998684e-01,9.999708e-01,9.999976e-01,1.000000e+00,2.550000e+02,1.090000e+02,1.420000e+02,3.600000e+01,2.000000e+00


In [13]:
all_frames.info()

<class 'pandas.DataFrame'>
Index: 1992567 entries, 0 to 9546
Columns: 125 entries, frame to p5_player demolished by
dtypes: float32(124), int64(1)
memory usage: 972.9 MB


In [12]:
all_frames.isna().sum().sum()

np.int64(0)

## Autocorrelation